# SynBioCrow 2.4.13.10 — Currency-Metabolite Boundary Audit\nStrict closure remains unchanged. A second diagnostic closure treats an explicit structural class of water/small currency/nucleotide-cofactor-like species as externally supplied, to distinguish bookkeeping burden from missing carbon chemistry.\n

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib,json,os,shutil,subprocess,sys,zipfile,time
drive.mount("/content/drive",force_remount=False)
T0=time.time()
REPO=Path("/content/SynBioCrow_2_4_13_10")
RUNOUT=Path("/content/synbiocrow_2_4_13_10")
OUT=Path("/content/drive/MyDrive/SynBioCrow/2.4/development_routes_2_4_13_10")
STATE=Path("/content/drive/MyDrive/SynBioCrow/2.4/state_2_4_13_10")
OUT.mkdir(parents=True,exist_ok=True); STATE.mkdir(parents=True,exist_ok=True)
def status(p,m): print(f"[2.4.13.10] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {m}",flush=True)
def run(cmd,cwd=None,timeout=3600,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    env=os.environ.copy()
    if cwd: env["PYTHONPATH"]=str(cwd)+os.pathsep+env.get("PYTHONPATH","")
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode: raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

status(5,"clone develop/2.4")
if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.4","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
status(12,"install core + DORAnet")
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[doranet]"],timeout=900)
status(20,"install pinned RetroBioCat2")
run([sys.executable,"-m","pip","install","-q","PyYAML","sqlalchemy","tables","gdown","tqdm"],timeout=900)
RBC2_REV="c5f32561b1ed99d0701a6dbb472dc448d6c082e8"
run([sys.executable,"-m","pip","install","-q","--no-deps","--no-build-isolation",f"git+https://github.com/willfinnigan/RetroBioCat-2.git@{RBC2_REV}"],timeout=1200)
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=600)

status(26,"RBC2 commercial database intentionally bypassed")
print("[2.4.13.10] RBC2 will use frozen sink-panel StartingMaterialEvaluator; no commercial source_mols.db required.",flush=True)

status(30,"strict backend readiness gate")
probe=run([sys.executable,"-m","synbiocrow.cli","readiness","--json"],cwd=REPO,timeout=300)
raw=probe.stdout
start=raw.find("{")
end=raw.rfind("}")
if start<0 or end<start:
    raise RuntimeError("readiness JSON object not found in mixed subprocess output")
readiness=json.loads(raw[start:end+1])
assert readiness["doranet"]["available"],readiness["doranet"]
assert readiness["retrobiocat2"]["available"],readiness["retrobiocat2"]
assert readiness["retrobiocat2"].get("native_mcts_api"),readiness["retrobiocat2"]
print("DORAnet + RetroBioCat2 readiness PASS",flush=True)

status(40,"generate development routes with frozen sink panel")
if RUNOUT.exists(): shutil.rmtree(RUNOUT)
run([sys.executable,str(REPO/"scripts/v24_generate_development_routes_2_4_13_10.py"),
 "--tranche",str(REPO/"benchmarks/2.4/source_tranche_v1.json"),
 "--split-manifest",str(REPO/"benchmarks/2.4/split_manifest_2_4_11.json"),
 "--sink-panel",str(REPO/"benchmarks/2.4/development_sink_panel_2_4_13_1.json"),
 "--out-dir",str(RUNOUT),"--state-dir",str(STATE),
 "--max-route-steps","8","--max-routes","100"],cwd=REPO,timeout=10800)

summary=json.loads((RUNOUT/"generation_summary.json").read_text())
assert summary["validation_truth_accessed"] is False
assert summary["tuning_performed"] is False
assert len(summary["targets"])==4
status(88,"persist outputs to Drive")
for p in RUNOUT.iterdir():
    if p.is_file(): shutil.copy2(p,OUT/p.name)
bundle=OUT/"SynBioCrow_2_4_13_10_CURRENCY_BOUNDARY_AUDIT_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=bundle: z.write(p,p.name)
status(100,"PASS")
print("bundle:",bundle)
print("sha256:",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))


## Rescue download


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/development_routes_2_4_13_10/SynBioCrow_2_4_13_10_CURRENCY_BOUNDARY_AUDIT_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
